# 🌳 Tree of Thought + Subquestion Decomposition

**Exercise duration: ~5 minutes** | Run every cell top-to-bottom

---

## What You'll Build

| Technique | Idea | Task |
|---|---|---|
| **Tree of Thought (ToT)** | Explore multiple reasoning paths, keep the best | Creative story writing |
| **Subquestion Decomposition** | Break hard questions into simpler sub-problems | Math word problems |
| **Plan-and-Solve** | Sketch a plan before executing | Math word problems |

---

**[RUN]** = just execute. **[TODO]** = fill in before running.

> ⚠️ Use a **GPU runtime**.

## 1. Setup

**[RUN]**

In [ ]:
!pip install -q transformers accelerate datasets
import torch
print(f"\u2705 GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else chr(10)+chr(10)+'  WARNING: No GPU found. Go to Runtime > Change runtime type > T4 GPU.'}")

In [ ]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
print(f"Loading {MODEL_ID} ...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,   # half-precision: 2x faster, half the VRAM
    device_map="auto",           # auto-selects GPU
)
model.eval()
print(f"Model loaded on {next(model.parameters()).device}")

In [ ]:
import uuid, time, html as html_lib
from IPython.display import display, HTML

def generate_response(messages, max_new_tokens=200, creative=False):
    """
    Send a chat-formatted message list to the model and return the response.

    creative=False  ->  greedy decoding (fast, deterministic).
                        Use for scoring, parsing, structured output.
    creative=True   ->  sampling at temperature=0.7.
                        Use for story generation or open-ended tasks.
    """
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=creative,
            temperature=0.7 if creative else 1.0,
            pad_token_id=tokenizer.eos_token_id,
        )
    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True
    )
    return response.strip()


def display_sample(question, answer):
    display(HTML(
        f"<p><strong>Question:</strong> {question}</p>"
        f"<p><strong>Answer:</strong> {answer}</p>"
    ))


def display_response(prompt_text, response_text, elapsed=None):
    uid = str(uuid.uuid4()).replace("-", "")
    time_tag = f"<div style='color:#666;font-size:90%;margin-top:4px'>&#x23F1; {elapsed:.1f}s</div>" if elapsed else ""
    safe_p = html_lib.escape(str(prompt_text).strip())
    safe_r = html_lib.escape(str(response_text).strip())
    display(HTML(
        "<style>.rt{border-collapse:collapse;width:100%;margin:10px 0;font-family:'Segoe UI',sans-serif;font-size:14px}"
        ".rt th,.rt td{border:1px solid #ddd;padding:9px 12px;vertical-align:top}"
        ".rt th{background:#f5f5f5;width:120px;font-weight:600}"
        "pre.rm{white-space:pre-wrap;margin:0}</style>"
        f"<table class='rt'>"
        f"<tr><th>Prompt</th><td><pre class='rm'>{safe_p}</pre></td></tr>"
        f"<tr><th>Response</th><td><pre class='rm'>{safe_r}</pre>{time_tag}</td></tr>"
        "</table>"
    ))


def run_and_display(messages, max_new_tokens=200, creative=False):
    """Generate and display in a table. Returns the response string."""
    prompt_text = messages[-1]["content"]
    t0 = time.time()
    response = generate_response(messages, max_new_tokens=max_new_tokens, creative=creative)
    elapsed = time.time() - t0
    display_response(prompt_text, response, elapsed)
    return response

## Part 1: Tree of Thought (ToT)

### What makes ToT different from CoT?

Standard Chain of Thought commits to one reasoning path. **Tree of Thought** lets the model branch out: generate *multiple candidate next steps*, score them, keep the best, repeat.

Here we apply ToT to **creative story writing**.

### Step 1: Configuration

**[RUN]** With `B=2, D=1` the ToT loop makes just **3 model calls** total. Increase `D` later if you want a deeper tree.

In [ ]:
from typing import List, Tuple
import random
random.seed(42)

B = 2   # branching factor: candidate beats per step
D = 1   # depth: number of outline steps before writing the story

**[RUN]** Define the prompt templates. Notice the `{placeholders}`.

In [ ]:
TASK_INSTRUCTION = (
    "Write a vivid, ~100-word short story for a general audience. "
    "It must have: a surprising twist, strong imagery, and an emotionally satisfying resolution. "
    "Use descriptive nouns instead of proper names."
)

PROPOSE_PROMPT = """
You are outlining a short story. The brief is:
{task}

Outline so far:
{outline}

Propose ONE next story beat in 1-2 sentences. Be concrete and evocative.
Return ONLY the beat, no explanation.
""".strip()

SCORE_PROMPT = """
Brief:
{task}

Outline so far:
{outline}

Candidate beat:
{candidate}

Score this beat 0-10 for coherence, creativity, and fit with the brief.
Return ONLY a single integer 0-10.
""".strip()

### Quick Python check: `.format()` fills placeholders

**[RUN]**

In [ ]:
# How .format() fills placeholders:
prompt = PROPOSE_PROMPT.format(task=TASK_INSTRUCTION, outline="(none yet)")
print(prompt)

### ✏️ [TODO] Try it for the scoring template

`SCORE_PROMPT` has **three** placeholders: `{task}`, `{outline}`, `{candidate}`. Call `.format()` with all three and print the result.

In [ ]:
# TODO: Call SCORE_PROMPT.format() with all three placeholders
# (task, outline, candidate) and print the result.


Good. You'll use this `.format()` pattern in every exercise below.

### Step 2: ToT Core Functions

**[IMPORTANT] Chat Template reminder:**

```python
messages = [{"role": "user", "content": "your prompt"}]
```

**Speed tip built into `generate_response`:**
- `creative=False` → greedy, fastest. Use for scoring (just need an integer).
- `creative=True` → sampled. Use for story generation.

### ✏️ [TODO] Implement `propose_next_beats` and `score_beats`

Two tasks:
1. **`propose_next_beats`**: call `generate_response(message, max_new_tokens=80, creative=True)` and wrap the result in a list `[response]`.
2. **`score_beats`**: build the scoring prompt, call `generate_response(message, max_new_tokens=8, creative=False)`, parse the integer score.

In [ ]:
def propose_next_beats(outline_beats, b):
    """Propose b candidate story beats for the current outline."""
    outline_text = "\n- ".join(outline_beats) if outline_beats else "(none yet)"
    prompt = PROPOSE_PROMPT.format(task=TASK_INSTRUCTION, outline=outline_text)
    message = [{"role": "user", "content": prompt}]
    # TODO: call generate_response(message, max_new_tokens=80, creative=True)
    # and return the result wrapped in a list: [response]
    pass


def score_beats(outline_beats, candidates):
    """Score each candidate beat. Returns list of (beat, float_score)."""
    scored = []
    outline_text = "\n- ".join(outline_beats) if outline_beats else "(none yet)"
    for candidate in candidates:
        # TODO: build the scoring prompt using SCORE_PROMPT.format()
        prompt = ""
        # TODO: message = [{"role": "user", "content": prompt}]
        message = []
        try:
            # Use max_new_tokens=8, creative=False (we just need one integer)
            resp = generate_response(message, max_new_tokens=8, creative=False)
            digits = "".join(ch for ch in resp if ch.isdigit())
            score = float(digits[:2]) if digits else 0.0
            score = max(0.0, min(10.0, score))
        except Exception:
            score = 0.0
        scored.append((candidate, score))
    return scored

### ✏️ [TODO] Step 3: Beam Search

The ToT loop: propose candidates → score them → keep the top `B`.

Fill in the two `TODO` lines.

In [ ]:
def beam_search_outline(b=B, d=D):
    """
    Tree-of-Thought beam search over story beats.
    State: (beats_so_far, running_avg_score).
    """
    from typing import List, Tuple
    beams = [([], 0.0)]   # list of (beats, avg_score)
    for step in range(1, d + 1):
        new_beams = []
        for beats, avg_score in beams:
            # TODO: call propose_next_beats(beats, b=b) to get candidates
            candidates = []
            # TODO: call score_beats(beats, candidates)
            scored = []
            for beat, s in scored:
                new_beams.append((beats + [beat], (avg_score * (step-1) + s) / step))
        new_beams.sort(key=lambda x: x[1], reverse=True)
        beams = new_beams[:b]
        print(f"  Step {step}: best score = {beams[0][1]:.1f}/10")
    return beams[0][0]

### ✏️ [TODO] Step 4: Compose the Final Story

Once we have the best outline beats, write the full story. Fill in the three `TODO` lines.

In [ ]:
COMPOSE_PROMPT = """
Using the outline below, write the final story (~100 words) in one block of prose.
Follow the brief closely.

Brief: {task}

Outline beats:
{beats}

Begin the story now.
""".strip()

def compose_story(beats):
    beats_text = "\n".join(f"- {b}" for b in beats)
    # TODO: build prompt with COMPOSE_PROMPT.format(task=..., beats=...)
    prompt = ""
    # TODO: message = [{"role": "user", "content": prompt}]
    message = []
    # TODO: call run_and_display(message, max_new_tokens=180, creative=True)

### Step 5: Run the Full ToT Pipeline

**[RUN]**

In [ ]:
print("Running Tree-of-Thought outline search...")
best_beats = beam_search_outline(b=B, d=D)

print("\nBest outline beats:")
for i, beat in enumerate(best_beats, 1):
    print(f"  {i}. {beat}")

print("\nComposing final story from outline...")
compose_story(best_beats)

### Baseline: Zero-Shot Story

**[RUN]** Same task, no ToT. Compare quality!

In [ ]:
BASELINE_PROMPT = """
{task}

Write the story now (~100 words) in one block of prose.
""".strip()

print("Zero-shot baseline (no ToT outline)...")
prompt = BASELINE_PROMPT.format(task=TASK_INSTRUCTION)
run_and_display([{"role": "user", "content": prompt}], max_new_tokens=180, creative=True)

---

## Part 2: Subquestion Decomposition

### The idea

Some problems are too complex to answer in one shot. Break them into smaller, focused sub-questions, answer each, combine the results.

**think → split → solve → combine**

**[RUN]** Load the GSM8K socratic split (answers written as step-by-step question chains).

In [ ]:
ds_socratic = load_dataset("openai/gsm8k", "socratic")
train_split_socratic = ds_socratic["train"]
test_split_socratic  = ds_socratic["test"]
print(f"Train: {len(train_split_socratic)} | Test: {len(test_split_socratic)}")
display_sample(train_split_socratic[0]["question"], train_split_socratic[0]["answer"])

### Few-shot: Show the model one example, then ask it to follow the same pattern.

**[RUN]**

In [ ]:
example = train_split_socratic[0]
test_q  = test_split_socratic[0]["question"]

few_shot_soc = (
    "Here is an example of solving by decomposing into subquestions:\n\n"
    f"Q: {example['question']}\nA: {example['answer']}\n\n"
    f"Now answer this question the same way:\nQ: {test_q}\nA:"
)
print(few_shot_soc[:500], "...")

In [ ]:
run_and_display([{"role": "user", "content": few_shot_soc}], max_new_tokens=200)

### ✏️ [TODO] Zero-shot Subquestion Decomposition

Same result, no example. Write a 2-3 sentence instruction telling the model to decompose the question into subquestions and answer them.

> 💡 Keep it short and direct.

In [ ]:
test_q = test_split_socratic[0]["question"]

# TODO: Write a short (2-3 sentence) zero-shot instruction telling the model to
# break the question into subquestions, answer each, then give the final answer.

prompt = ""   # TODO: your prompt here (must include test_q)
run_and_display([{"role": "user", "content": prompt}], max_new_tokens=200)

---

## Part 3: Plan-and-Solve

### The idea

Separate *planning* from *execution*:
1. **Plan** 🗺️ — identify what's given and outline the steps
2. **Solve** 🧮 — execute the plan systematically

### Few-shot Plan-and-Solve

**[RUN]**

In [ ]:
test_q = test_split_socratic[0]["question"]

few_shot_PS = (
    "Follow this format to solve the question:\n\n"
    "Q: James runs 3 sprints 3 times a week, 60 meters each. Total meters per week?\n"
    "A: Given: 3 sprints x 3 times = 9 sprints/week. Each = 60 m.\n"
    "   Plan: multiply total sprints by distance.\n"
    "   Calculation: 9 x 60 = 540 m.\n"
    "   Answer: 540 meters per week.\n\n"
    f"Q: {test_q}\nA:"
)
print(few_shot_PS)

In [ ]:
run_and_display([{"role": "user", "content": few_shot_PS}], max_new_tokens=200)

### ✏️ [TODO] Zero-shot Plan-and-Solve

Write an instruction directing the model to: (1) list what's given, (2) write a plan, (3) execute it, (4) state the answer.

In [ ]:
test_q = test_split_socratic[0]["question"]

# TODO: Write an instruction that tells the model to:
# 1. List what is given
# 2. Write a plan
# 3. Execute it step by step
# 4. State the final answer

prompt = ""   # TODO: your prompt here (must include test_q)
run_and_display([{"role": "user", "content": prompt}], max_new_tokens=200)

# 🏁 Key Takeaways

1. **Zero-shot reasoning is powerful** — a well-crafted instruction teaches the model the *format* of reasoning without any examples.

2. **Reasoning techniques improve both accuracy and interpretability** — ToT, Subquestion Decomposition, and Plan-and-Solve all make the model's thinking *visible*.

3. **LLMs reason better when structured** — the model isn't 'smarter'; it's given a better framework to operate within.

> 🔮 **Next up:** Notebook 3 uses the model to *automatically discover* better prompts through OPRO.